# Week 29, Day 2: The resting potential: the Goldman-Hodgkin-Katz equation

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w029/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Resting and peak potentials

Write `ghk_voltage(pK, pNa, pCl)` for yesterday's mammalian concentrations. Compute the resting potential `v_rest` for permeability ratios 1 : 0.05 : 0 and the peak potential `v_peak` when, during a spike, the ratio becomes 1 : 20 : 0. Then see what adding chloride permeability (0.45) does to rest.

In [ ]:
import numpy as np
VT = 26.725                                     # mV at 37 C
Ko, Ki, Nao, Nai, Clo, Cli = 5, 140, 145, 12, 110, 7

def ghk_voltage(pK, pNa, pCl):
    return ...

v_rest = ghk_voltage(1, 0.05, 0)
v_peak = ...
print(f"rest {v_rest:.1f} mV;  peak {v_peak:.1f} mV;  rest with chloride {ghk_voltage(1, 0.05, 0.45):.1f} mV")
print(f"compare E_K = {VT * np.log(Ko / Ki):.1f} mV and E_Na = {VT * np.log(Nao / Nai):.1f} mV")

In [ ]:
import numpy as np
def test_ghk():
    "ghk_voltage gives V_T ln(numerator / denominator), and reduces to the Nernst potential of potassium when only potassium permeates"
    assert abs(ghk_voltage(1, 0, 0) - 26.725 * np.log(5 / 140)) < 1e-9
    assert abs(ghk_voltage(1, 0.05, 0.45) - 26.725 * np.log((5 + 0.05 * 145 + 0.45 * 7) / (140 + 0.05 * 12 + 0.45 * 110))) < 1e-9

def test_values():
    "rest is near -65 mV and the peak, with sodium 20 times more permeable, near +54 mV"
    assert -68 < v_rest < -62 and 50 < v_peak < 58

test_ghk()
test_values()
print("All checks passed")

### Exercise: Fitting the permeability ratio

The array `V` holds resting potentials measured (with 1 mV of noise) at eight external potassium concentrations, for squid concentrations [K]in = 400, [Na]in = 50 and [Na]out = 440 mM. Write `model(Ko, b)`, the GHK resting potential for ratio b = P_Na / P_K, and find the ratio `b_fit` on `grid` that minimises the sum of squared errors.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
VTs = 1000 * 8.314 * 279.45 / 96485            # thermal voltage at 6.3 C, mV
Ki, Nai, Nao = 400, 50, 440
Ko = np.array([2.5, 5, 10, 20, 50, 100, 200, 450])
V = np.array([-71.6, -67.9, -64.3, -58.6, -42.8, -29.5, -15.4, 2.0])

def model(Ko, b):
    return ...

grid = np.linspace(0, 0.2, 2001)
sse = [np.sum((V - model(Ko, b)) ** 2) for b in grid]
b_fit = ...
print(f"P_Na / P_K = {b_fit:.3f};  slope at high K: {VTs * np.log(10):.1f} mV per decade")
plt.semilogx(Ko, V, "o", label="measured"); plt.semilogx(Ko, model(Ko, b_fit), label="GHK fit")
plt.semilogx(Ko, VTs * np.log(Ko / Ki), "--", label="E_K"); plt.xlabel("[K]out (mM)"); plt.ylabel("V (mV)"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_model():
    "model(Ko, b) gives the GHK resting potential for squid concentrations with P_Na / P_K = b"
    assert abs(model(20, 0.0) - VTs * np.log(20 / 400)) < 1e-9 and abs(model(20, 0.1) - VTs * np.log((20 + 44) / (400 + 5))) < 1e-9

def test_fit():
    "the fitted ratio minimises the squared error over the grid and is between 0.02 and 0.07"
    sse = [np.sum((V - model(Ko, b)) ** 2) for b in grid]
    assert abs(b_fit - grid[np.argmin(sse)]) < 1e-12 and 0.02 < b_fit < 0.07

test_model()
test_fit()
print("All checks passed")

### Exercise: The currents at rest

Write `ghk_current(V, P, c_in, c_out, z=1)`, the GHK current in units of F (positive outward). Evaluate the potassium current `i_K` (P = 1) and sodium current `i_Na` (P = 0.05) at the resting potential `v_rest` from the cell above.

In [ ]:
import numpy as np
VT = 26.725

def ghk_current(V, P, c_in, c_out, z=1):
    u = z * V / VT
    return ...

i_K = ghk_current(v_rest, 1, 140, 5)
i_Na = ...
print(f"at {v_rest:.1f} mV: potassium {i_K:+.2f}, sodium {i_Na:+.2f}, total {i_K + i_Na:+.1e}  (units of P_K F mM)")

In [ ]:
import numpy as np
def test_current():
    "ghk_current gives P u (c_in - c_out e^-u) / (1 - e^-u) with u = z V / V_T"
    u = -65 / VT
    assert abs(ghk_current(-65, 1, 140, 5) - u * (140 - 5 * np.exp(-u)) / (1 - np.exp(-u))) < 1e-9

def test_balance():
    "at rest the potassium current is outward, the sodium current inward, and they cancel"
    assert i_K > 0 and i_Na < 0 and abs(i_K + i_Na) < 1e-9

test_current()
test_balance()
print("All checks passed")